In [ ]:
"""
session_summary.ipynb

plot model metrics across sessions

Author: Stellina X. Ao
Created: 2026-05-04
Last Modified: 2026-05-06
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import pickle
import numpy as np
import matplotlib.pyplot as plt
from utils.paths import FIGURES_DIR, MODELS_DIR

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

## LVM - tributaries

In [ ]:
from core.data import subject_ids, session_ids

epochs = [
    {"key": "choice", "alignment": "choice", "tpre": 0.5, "tpost": 0.5},
    {"key": "reward", "alignment": "reward", "tpre": 0, "tpost": 1},
    {"key": "iti", "alignment": "trial_start", "tpre": 1.5, "tpost": -0.5},
]
epochs_key = [epoch["key"] for epoch in epochs]

regions = ["all", "ACC", "M2", "DMS", "DLS"]
strategies = ["both", "mb", "mf"]

n_cv = 5

In [ ]:
def get_metrics(subj_id, reg, strategy, metric):
    sess_ids = session_ids[np.where(subject_ids == subj_id)[0][0]]
    metrics = {epoch: np.full((len(sess_ids), n_cv), np.nan) for epoch in epochs_key}

    for i, sess_id in enumerate(sess_ids):
        print(sess_id)
        file_path = (
            MODELS_DIR
            / "fit"
            / subj_id
            / sess_id
            / "river_n_tributaries"
            / "results_dict.pkl"
        )

        if not file_path.is_file():
            continue

        with open(file_path, "rb") as f:
            res_dict = pickle.load(f)

        for epoch in epochs_key:
            print(epoch)
            if strategy == "both":
                families = res_dict[reg][epoch][strategy]["families"]
                if len(families) == 0:
                    # region doesn't exist for this session
                    break
            else:
                try:
                    families = res_dict[reg][epoch][strategy]["families"]
                except KeyError:
                    # region doesn't exist for this session
                    break

            for j, family in enumerate(families):
                if metric == "qi":
                    metric_ = family.qi
                elif metric == "r2test_taskvar":
                    try:
                        metric_ = family.res_taskvar["r2test"].mean()
                    except AttributeError:
                        metric_ = np.nan
                elif metric == "r2test_affine":
                    try:
                        metric_ = family.res_affine["r2test"].mean()
                    except AttributeError:
                        metric_ = np.nan
                metrics[epoch][i][j] = metric_
    return metrics

In [ ]:
def get_metrics_3x3(subj_id, metric):
    metrics = {}
    for reg in regions:
        print(f"! {reg}")
        metrics[reg] = {}
        for strategy in strategies:
            print(f"> {strategy}")
            metrics[reg][strategy] = {}

            metrics[reg][strategy] = get_metrics(subj_id, reg, strategy, metric)
            print(metrics[reg][strategy])

        if np.array(
            [
                np.isnan(metrics[reg][strategy][epoch])
                for epoch in epochs_key
                for strategy in strategies
            ]
        ).all():
            metrics.pop(reg, None)
    return metrics

In [ ]:
from core.data import colors_epoch
from scipy.stats import sem


def plot_metrics_3x3(subj_id, metrics, metric, do_save=True):
    fig, axes = plt.subplots(ncols=len(metrics), nrows=3, figsize=(5, 4), sharey="all")

    for i, reg in enumerate(metrics):
        for j, strategy in enumerate(metrics[reg]):
            for epoch in metrics[reg][strategy]:
                metrics_epoch = metrics[reg][strategy][epoch]

                metrics_avg = np.nanmean(metrics_epoch, axis=1)
                metrics_sem = sem(metrics_epoch, axis=1, nan_policy="omit")

                session_idxs = np.arange(metrics_epoch.shape[0])

                axes[j][i].plot(
                    session_idxs, metrics_avg, color=colors_epoch[epoch], label=epoch
                )
                axes[j][i].fill_between(
                    session_idxs,
                    metrics_avg - metrics_sem,
                    metrics_avg + metrics_sem,
                    color=colors_epoch[epoch],
                    alpha=0.5,
                )
            axes[j][i].set_xlabel("Sessions")
            axes[j][i].set_ylabel(metric)
            axes[j][i].legend()

    fig.tight_layout()
    if do_save:
        from utils.paths import FIGURES_DIR

        fpath_png = FIGURES_DIR / "reg_strategy_epoch" / subj_id / f"{metric}.png"
        fpath_svg = FIGURES_DIR / "reg_strategy_epoch" / subj_id / f"{metric}.svg"
        fpath_png.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(fpath_png, dpi=300, bbox_inches="tight")
        fig.savefig(fpath_svg, dpi=300, bbox_inches="tight")

In [ ]:
def plot_metrics_3x3_bar(subj_id, metrics, metric, do_save=True):
    fig, axes = plt.subplots(
        ncols=len(metrics), nrows=3, figsize=(5, 4), sharex="all", sharey="all"
    )

    for i, reg in enumerate(metrics):
        for j, strategy in enumerate(metrics[reg]):
            epochs = metrics[reg][strategy].keys()
            metrics_avg = [
                np.nanmean(metrics[reg][strategy][epoch])
                for epoch in metrics[reg][strategy]
            ]
            metrics_sem = [
                sem(metrics[reg][strategy][epoch], nan_policy="omit", axis=None)
                for epoch in metrics[reg][strategy]
            ]

            axes[j][i].bar(
                epochs,
                metrics_avg,
                color=[colors_epoch[epoch] for epoch in metrics[reg][strategy]],
            )
            axes[j][i].errorbar(
                epochs, metrics_avg, metrics_sem, color="k", fmt=".", capsize=2
            )
            axes[j][i].set_ylabel(metric)

    fig.tight_layout()
    if do_save:
        from utils.paths import FIGURES_DIR

        fpath_png = FIGURES_DIR / "reg_strategy_epoch" / subj_id / f"{metric}_bars.png"
        fpath_svg = FIGURES_DIR / "reg_strategy_epoch" / subj_id / f"{metric}_bars.svg"
        fpath_png.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(fpath_png, dpi=300, bbox_inches="tight")
        fig.savefig(fpath_svg, dpi=300, bbox_inches="tight")

In [ ]:
def plot_metrics_3x3_hist(subj_id, metrics, metric, do_save=True):
    fig, axes = plt.subplots(
        ncols=len(metrics), nrows=3, figsize=(5, 4), sharex="all", sharey="all"
    )

    for i, reg in enumerate(metrics):
        for j, strategy in enumerate(metrics[reg]):
            ax = axes[j][i]
            epochs = metrics[reg][strategy].keys()

            for epoch in epochs:
                metrics_epoch = metrics[reg][strategy][epoch]

                ax.hist(
                    np.ravel(metrics_epoch),
                    bins=np.linspace(0, 0.8, 25),
                    color=colors_epoch[epoch],
                    histtype="stepfilled",
                    alpha=0.5,
                )

            ax.set_xlabel(metric)
            ax.set_ylabel("freq")

    fig.tight_layout()
    if do_save:
        from utils.paths import FIGURES_DIR

        fpath_png = FIGURES_DIR / "reg_strategy_epoch" / subj_id / f"{metric}_hist.png"
        fpath_svg = FIGURES_DIR / "reg_strategy_epoch" / subj_id / f"{metric}_hist.svg"
        fpath_png.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(fpath_png, dpi=300, bbox_inches="tight")
        fig.savefig(fpath_svg, dpi=300, bbox_inches="tight")

In [ ]:
subj_id = "MR82"
metric = "r2test_taskvar"

# save_path = MODELS_DIR / "fit" / subj_id / "metrics" / "no_cid_enforcement"/ f"{metric}.pkl"
save_path = MODELS_DIR / "fit" / subj_id / "metrics" / f"{metric}.pkl"
if save_path.is_file():
    with open(save_path, "rb") as f:
        metrics = pickle.load(f)

In [ ]:
plot_metrics_3x3_hist(subj_id, metrics, metric, do_save=True)

## Encoder - two strategies

In [ ]:
def load_family_strategy(subj_id, sess_id, region, seed):
    file_path = (
        MODELS_DIR
        / "fit"
        / subj_id
        / sess_id
        / "separate_strategy"
        / "encoder_no_update_cid"
        / "results_dict.pkl"
    )

    if not file_path.is_file():
        return

    with open(file_path, "rb") as f:
        res_dict = pickle.load(f)

    family_mb = res_dict[region]["mb"]["families"][seed]
    family_mf = res_dict[region]["mf"]["families"][seed]

    return family_mb, family_mf

In [ ]:
from sg.fitter import LVMFamily

subj_id = "MR82"
sess_id = "20251027_152036"
reg = "DLS"
seed = 0

family = LVMFamily(
    subj_id=subj_id,
    sess_id=sess_id,
    n_latents_mult=1,
    n_latents_addt=1,
    sanity_check=0,
    task_vars=[
        "response",
        "rewarded",
        "block_side",
        "response_prev",
        "rewarded_prev",
    ],
    n_splines=5,
    tpre=0.5,
    tpost=1,
)
family.fit_all()
family.eval()

In [ ]:
from squiggs.renderers import PETHRasterRenderer
from squiggs.neuron_viewer import NeuronViewer
from core.data import get_psths_cond, get_choice_ts
from pathlib import Path


def plot_psths_renderers(family, mode, reg, strategy):
    renderer = PETHRasterRenderer(
        event_times=get_choice_ts(family.trial_data, mode=mode),
        spike_times=family.spike_times[reg],
        peths=get_psths_cond(family.psths[reg], family.trial_data, mode=mode),
        pres=0.5,
        posts=1,
        binwidth_s=25 / 1000,
        s=0.5,
        linewidths=0.5,
        save_subdir=Path("peths") / subj_id / sess_id / reg / mode / strategy,
    )

    _ = NeuronViewer(
        num_units=family.psths[reg].shape[0], render_func=renderer, fig_dir=FIGURES_DIR
    )

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"
reg = "DLS"
seed = 0

family_mb, family_mf = load_family_strategy(subj_id, sess_id, reg, seed)

In [ ]:
def plot_beta_tv(subj_id, sess_id, tv, region, seed, color=False):
    family_mb, family_mf = load_family_strategy(subj_id, sess_id, region, seed)

    beta_mb = family_mb.mod_taskvar.tv.weight.data[:]
    beta_mf = family_mf.mod_taskvar.tv.weight.data[:]

    tv_idxs = []
    tv_labels = []
    counter = 0
    for tv_ in family_mb.task_vars:
        for val in family_mb.trial_data[tv_].unique():
            if tv_ == tv:
                tv_idxs.append(counter)
                tv_labels.append(f"{tv_}_{val}")
            counter += 1

    fig, axes = plt.subplots(nrows=1, ncols=len(tv_idxs))

    for i, ax in enumerate(axes.flat):
        if color:
            c = ax.scatter(
                beta_mb[tv_idxs[i]],
                beta_mf[tv_idxs[i]],
                s=0.5,
                cmap="hsv",
                c=np.arange(beta_mb.shape[1]),
            )
            fig.colorbar(c)
        else:
            ax.scatter(beta_mb[tv_idxs[i]], beta_mf[tv_idxs[i]], s=0.5, color="#17612F")
        ax.set_xlabel(r"mb $\beta$")
        ax.set_ylabel(r"mf $\beta$")
        ax.set_title(f"{tv_labels[i]}")
    fig.tight_layout()


plot_beta_tv(subj_id, sess_id, "response", reg, seed, color=False)